# Memory, state and sessions

Read the [guide](README.md) for prerequisites, costs and exercises. Run all cells in order with the `notebooks`, `rag`, `memory` and `aws` groups installed. Calls use offline fixtures, not live model judgments.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_state_and_messages

[Script](examples/01_state_and_messages.py). Predict the result before running.


In [ ]:
"""Distinguish private application state from messages sent to a model.

Run: uv run python modules/09_memory/examples/01_state_and_messages.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    args, cfg = lesson_settings()
    agent = Agent(
        model=get_model(cfg, responses=["Please tell me your seat preference."]),
        callback_handler=None,
    )
    # KEY IDEA: state is not automatically a prompt; explicitly select any facts to expose.
    agent.state.set("seat", "aisle")
    result = agent(
        "Can you see a preference that the application has not sent?", limits={"turns": 2}
    )
    console.print("Private state:", agent.state.get("seat"))
    console.print("Message roles:", [m["role"] for m in agent.messages])
    complete(result, cfg)


run_lesson(main, "modules/09_memory/examples/01_state_and_messages.py")

## 02_file_sessions

[Script](examples/02_file_sessions.py). Predict the result before running.


In [ ]:
"""Resume file sessions with an application-derived customer namespace.

Run: uv run python modules/09_memory/examples/02_file_sessions.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

import tempfile
from pathlib import Path

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.preferences import Preferences
from common.pretty import console, run_cli


def main() -> None:
    args, cfg = lesson_settings()
    with tempfile.TemporaryDirectory(prefix="nimbus-sessions-") as directory:
        preferences = Preferences(Path(directory))
        first_manager = preferences.session("CUS-001", "trip")
        first = Agent(
            model=get_model(cfg, responses=["Lisbon recorded."]),
            callback_handler=None,
            session_manager=first_manager,
        )
        first("I am planning Lisbon. Acknowledge in one short sentence.", limits={"turns": 2})
        # KEY IDEA: a new Agent restores history through the same scoped session ID.
        resumed = Agent(
            model=get_model(cfg, responses=["Lisbon."]),
            callback_handler=None,
            session_manager=preferences.session("CUS-001", "trip"),
        )
        console.print("Restored messages:", len(resumed.messages))
        result = resumed("Which city?", limits={"turns": 2})
        other = preferences.session("CUS-002", "trip")
        console.print("Different customer session:", other.session_id != first_manager.session_id)
        complete(result, cfg)


run_lesson(main, "modules/09_memory/examples/02_file_sessions.py")

## 03_s3_sessions

[Script](examples/03_s3_sessions.py). Predict the result before running.


In [ ]:
"""Round-trip S3 sessions with moto; live mode uses an existing tagged bucket.

Run: uv run python modules/09_memory/examples/03_s3_sessions.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

from contextlib import nullcontext

import boto3
from moto import mock_aws
from strands import Agent
from strands.session.s3_session_manager import S3SessionManager

from common.config import course_value, load_settings
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    args, cfg = lesson_settings()
    target = load_settings(provider="bedrock")
    context = mock_aws() if args.mock else nullcontext()
    with context:
        session = boto3.Session(
            region_name=target.aws_region,
            **(
                {"aws_access_key_id": "testing", "aws_secret_access_key": "testing"}
                if args.mock
                else {"profile_name": target.aws_profile or None}
            ),
        )
        bucket = "nimbus-session-fixture" if args.mock else course_value("NIMBUS_SESSION_BUCKET")
        if not bucket:
            raise ValueError("Set NIMBUS_SESSION_BUCKET after deploying the optional session lab")
        if args.mock:
            session.client("s3").create_bucket(
                Bucket=bucket,
                **(
                    {}
                    if target.aws_region == "us-east-1"
                    else {"CreateBucketConfiguration": {"LocationConstraint": target.aws_region}}
                ),
            )

        def manager():
            return S3SessionManager(
                "demo-trip", bucket=bucket, prefix="CUS-001", boto_session=session
            )

        first = Agent(
            model=get_model(cfg, responses=["Budget saved in this session."]),
            session_manager=manager(),
            callback_handler=None,
        )
        first("My budget is 600 dollars.", limits={"turns": 2})
        # KEY IDEA: a new manager reads the persisted messages; moto exercises actual S3 requests.
        restored = Agent(
            model=get_model(cfg, responses=[] if args.mock else None),
            session_manager=manager(),
            callback_handler=None,
        )
        assert len(restored.messages) == 2
        console.print("S3 restored messages:", len(restored.messages))
        manager().delete_session("demo-trip")
        console.print("Demo session deleted; bucket cleanup belongs to the infrastructure lab.")
    complete()


run_lesson(main, "modules/09_memory/examples/03_s3_sessions.py")

## 04_local_preferences

[Script](examples/04_local_preferences.py). Predict the result before running.


In [ ]:
"""Store consented bounded preferences, expire them, and delete the user footprint.

Run: uv run python modules/09_memory/examples/04_local_preferences.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

import tempfile
from pathlib import Path

from common.lessons import complete, lesson_settings
from common.preferences import Preferences
from common.pretty import console, run_cli


def main() -> None:
    args, cfg = lesson_settings()
    with tempfile.TemporaryDirectory(prefix="nimbus-preferences-") as directory:
        store = Preferences(Path(directory))
        # KEY IDEA: a trusted application identity and consent are outside model arguments.
        store.save("CUS-001", "seat", "aisle", consent=True, now=0)
        store.save("CUS-002", "meal", "vegetarian", consent=True, now=86400)
        console.print("CUS-001:", store.read("CUS-001", now=1))
        console.print("Expired at day 90:", store.read("CUS-001", now=90 * 86400))
        store.delete_user("CUS-001")
        console.print("Other traveler remains:", store.read("CUS-002", now=90 * 86400))
    complete()


run_lesson(main, "modules/09_memory/examples/04_local_preferences.py")

## 05_agentcore_memory

[Script](examples/05_agentcore_memory.py). Predict the result before running.


In [ ]:
"""Optional AgentCore adapter: scoped recall with persistence disabled in this lesson.

Run: uv run --group aws python modules/09_memory/examples/05_agentcore_memory.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

import asyncio
import json

from strands import Agent

from common.lessons import complete, lesson_settings
from common.memory_adapters import agentcore_manager
from common.models import get_model
from common.pretty import run_cli


def main() -> None:
    args, cfg = lesson_settings()
    with agentcore_manager(mock=args.mock) as manager:
        # KEY IDEA: a trusted actor and namespace constrain which memories are retrieved.
        memories = asyncio.run(manager.search("seat preference"))
        agent = Agent(
            model=get_model(cfg, responses=["Your recorded preference is an aisle seat."]),
            memory_manager=manager,
            callback_handler=None,
            retry_strategy=None,
            system_prompt="Answer only from the supplied memories. If empty, say you do not know.",
        )
        result = agent(
            json.dumps(
                {"question": "What seat do I prefer?", "memories": [m.content for m in memories]}
            ),
            limits={"turns": 2},
        )
        complete(result, cfg)


run_lesson(main, "modules/09_memory/examples/05_agentcore_memory.py")

## 06_mem0_store

[Script](examples/06_mem0_store.py). Predict the result before running.


In [ ]:
"""Use the supported Mem0 store and MemoryManager, then delete the scoped backend data.

Run: uv run --group memory python modules/09_memory/examples/06_mem0_store.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

import asyncio
import tempfile
from pathlib import Path

from strands.memory import MemoryManager

from common.lessons import complete, lesson_settings
from common.memory_adapters import mem0_store
from common.preferences import user_key
from common.pretty import console, run_cli


def main() -> None:
    args, cfg = lesson_settings()
    with tempfile.TemporaryDirectory(prefix="nimbus-mem0-") as directory:
        store, backend = mem0_store("CUS-001", mock=args.mock, directory=Path(directory))
        try:
            # KEY IDEA: explicit facts avoid unbounded automatic conversation extraction.
            manager = MemoryManager(stores=[store], injection=False)
            asyncio.run(manager.add("Traveler prefers vegetarian meals."))
            console.print("Scoped recall:", asyncio.run(manager.search("meal preference")))
            backend.delete_all(user_id=user_key("CUS-001"))
            assert not asyncio.run(manager.search("meal preference"))
            console.print("Mem0 scoped deletion verified.")
        finally:
            if not args.mock:
                backend.close()
                backend.vector_store.client.close()
    complete()


run_lesson(main, "modules/09_memory/examples/06_mem0_store.py")

## preferences_assistant

[Script](use_case/preferences_assistant.py). Predict the result before running.


In [ ]:
"""Remember explicit traveler preferences across new sessions and demonstrate deletion.

Run: uv run python modules/09_memory/use_case/preferences_assistant.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

import tempfile
from pathlib import Path

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.preferences import Preferences
from common.pretty import console, run_cli


def main() -> None:
    args, cfg = lesson_settings()
    with tempfile.TemporaryDirectory(prefix="nimbus-traveler-") as directory:
        store = Preferences(Path(directory))
        for name, value in {"seat": "aisle", "meal": "vegetarian", "budget_usd": "600"}.items():
            store.save("CUS-001", name, value, consent=True)
        # KEY IDEA: select only approved preference fields for the new session.
        selected = store.read("CUS-001")
        agent = Agent(
            model=get_model(
                cfg, responses=["Aisle, vegetarian meals, budget $600. Please confirm dates."]
            ),
            session_manager=store.session("CUS-001", "new-trip"),
            callback_handler=None,
            system_prompt=f"Nimbus planner. Confirm these possibly stale preferences: {selected}",
        )
        result = agent("Start a new trip.", limits={"turns": 2})
        complete(result, cfg)
        store.delete_user("CUS-001")
        assert store.read("CUS-001") == {}
        console.print("Traveler preferences and registered sessions deleted.")


run_lesson(main, "modules/09_memory/use_case/preferences_assistant.py")

## Practice

Complete the [exercises](README.md#exercises) and [quiz](quiz.md), then run the module tests.
